# Proceso de Machine Learning

## Selección del mejor Dataset

In [10]:
import pandas as pd
from pickle import dump

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

import warnings
from sklearn.exceptions import FitFailedWarning, DataConversionWarning

#warnings.filterwarnings("ignore", category=DataConversionWarning)
#warnings.filterwarnings("ignore", category=FitFailedWarning)
#warnings.filterwarnings("ignore", category=UserWarning)

X_train_CON_outliers = pd.read_excel("../data/processed/X_train_CON_outliers.xlsx")
X_train_SIN_outliers = pd.read_excel("../data/processed/X_train_SIN_outliers.xlsx")


y_train = pd.read_excel("../data/processed/y_train.xlsx")

y_train = y_train.squeeze()


datasets = [X_train_CON_outliers,
    X_train_SIN_outliers
    ]
models = []
metrics = []


for dataset in datasets:

    model = XGBClassifier(n_estimators = 5, random_state = 10)
    model.fit(dataset, y_train)
    y_pred = model.predict(dataset)
    metric = accuracy_score(y_train, y_pred)
    metrics.append(metric)
    models.append(model)


best_metric = max(metrics)
best_index = metrics.index(best_metric)
print(f"La mejor métrica es: \n{best_metric}")
print(f"El mejor dataset es: \n{datasets[best_index]}")

La mejor métrica es: 
0.9104234527687296
El mejor dataset es: 
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin    BMI  \
0            0.0      162             76             56    100.0  50.55   
1            2.0       87              0             23      0.0  28.90   
2            0.0      137             68             14    148.0  24.80   
3           10.0      115              0              0      0.0  35.30   
4            0.0      104             64             37     64.0  33.60   
..           ...      ...            ...            ...      ...    ...   
609          1.0      133            102             28    140.0  32.80   
610          4.0      129             60             12    231.0  27.50   
611          3.0      116             74             15    105.0  26.30   
612          1.0       88             30             42     99.0  50.55   
613          5.0       96             74             18     67.0  33.60   

     DiabetesPedigreeFunction  Age  

## Optimizacion de hiperparámetros

In [11]:
params = {
"n_estimators": [100, 200, 400],
"learning_rate": [0.01, 0.1],
"subsample": [0.6, 0.8, 1.0],
"max_depth": [3, 5]
}

### Grid search

In [12]:
grid = GridSearchCV(XGBClassifier(random_state=10), params, scoring="accuracy", verbose = 1)
grid.fit(datasets[best_index], y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
print("El mejor estimador es: ", best_model)
print("La mejor puntuación es: ", grid.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}
El mejor estimador es:  XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)
La mejor puntuación es:  0.7801546048247368


### Random search

In [13]:
grid = RandomizedSearchCV(XGBClassifier(random_state=10), params, scoring="accuracy", n_iter = 30)
grid.fit(datasets[best_index], y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
print(best_model)
print("El mejor estimador es: ", best_model)
print("La mejor puntuación es: ", grid.best_score_)

{'subsample': 0.8, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.01}
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)
El mejor estimador es:  XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              cols

## Métricas finales

Volvemos a entrenar el modelo con el mejor dataset de x_train y x_test

In [14]:
X_test_CON_outliers = pd.read_excel("../data/processed/X_test_CON_outliers.xlsx")
X_test_SIN_outliers = pd.read_excel("../data/processed/X_test_SIN_outliers.xlsx")


y_test = pd.read_excel("../data/processed/y_test.xlsx")

y_test = y_test.squeeze()

datasets_test = [X_test_CON_outliers,
    X_test_SIN_outliers
    ]


model_f = XGBClassifier(learning_rate=0.01, n_estimators=400,
                           random_state=10, subsample=0.8)
model_f.fit(datasets[best_index], y_train)

y_pred = model_f.predict(datasets[best_index])
metric_train = accuracy_score(y_train, y_pred)


y_pred = model_f.predict(datasets_test[best_index])
metric_test = accuracy_score(y_test, y_pred)


print(f"La mejor métrica de nuestros datasets x_train son: {metric_train} y los x_test son: {metric_test}")

#Guardado de modelo

dump(model, open("../models/modelo_entrenado_Boosting_Algorithm.sav", "wb"))

La mejor métrica de nuestros datasets x_train son: 0.9511400651465798 y los x_test son: 0.7662337662337663


## Conclusiones

In [18]:
models = [DecisionTreeClassifier(ccp_alpha=0.01, criterion='entropy', max_depth=8, random_state=10), RandomForestClassifier(criterion='entropy', max_depth=5, max_features='log2',
                       min_samples_split=4, n_estimators=200, random_state=10), XGBClassifier(learning_rate=0.01, n_estimators=400, random_state=10, subsample=0.8)]
metrics = []


for modelo in models:

    acc_modelo = modelo
    acc_modelo.fit(datasets[best_index], y_train)

    y_pred = acc_modelo.predict(datasets[best_index])
    metric_train = accuracy_score(y_train, y_pred)


    y_pred = acc_modelo.predict(datasets_test[best_index])
    metric_test = accuracy_score(y_test, y_pred)

    metrics.append([metric_train, metric_test])

nombres_modelos = ["Decision Tree", "Random Forest", "Boosting Algorithm"]

dicc_metricas = dict(zip(nombres_modelos, metrics))

print(dicc_metricas)


{'Decision Tree': [0.7947882736156352, 0.7662337662337663], 'Random Forest': [0.8403908794788274, 0.7727272727272727], 'Boosting Algorithm': [0.9511400651465798, 0.7662337662337663]}


Se puede apreciar que con los modelos hiperoptimizados entrenados en ejercicios anteriores, el Boosting Algorithm sigue dando mejores resultados a comparacion del Random Forest y el Decision Tree, siendo este  
último el que peores métricas tiene y eligiendo, para este caso, el Boosting Algorithm.